# Ion Channel Model Simulation Example with OBI-One Form Logic

This notebook demonstrates how to run an ion channel model simulation using the OBI-One form-based workflow, similar to the circuit simulation example.

In [ ]:
from entitysdk import Client
from obi_auth import get_token

from obi_notebook import get_projects
from obi_notebook import get_entities
import obi_one as obi

token = get_token(environment="production", auth_mode="daf")
project_context = get_projects.get_projects(token)

db_client = Client(environment="production", project_context=project_context, token_manager=token)

In [ ]:
# Optional: Download using unique ID
entity_ID = "<ICM-ID>"  # <<< FILL IN UNIQUE ION CHANNEL MODEL ID HERE

if entity_ID != "<ICM-ID>":
    ion_channel_model_ids = [entity_ID]
else:
# Alternative: Select from a table of entities
    ion_channel_model_ids = []
    ion_channel_model_ids = get_entities.get_entities("ion-channel-model", token, ion_channel_model_ids,
                                            project_context=project_context,
                                            multi_select=False,
                                            default_scale="small")



Here we load some more classes that we need to define our simulation scan configuration

In [7]:
from obi_one.core.block_subunit.complex_variable_holder import DurationVoltageCombination
from obi_one.core.info import Info
from obi_one.scientific.blocks.ion_channel_model import IonChannelModelWithConductance, IonChannelModelWithMaxPermeability, IonChannelModelWithoutConductance
from obi_one.scientific.blocks.recording import IonChannelVariableForRecording, IonChannelVariableRecording
from obi_one.scientific.blocks.stimuli.stimulus import MultiLevelSEClampSomaticStimulus
from obi_one.scientific.from_id.ion_channel_model_from_id import IonChannelModelFromID
from obi_one.scientific.tasks.generate_simulations.config.ion_channel_models import IonChannelModelSimulationScanConfig

We will have to get the correct type of ion channel to feed to the configuration:

In [8]:
from entitysdk.models.ion_channel_model import IonChannelModel

conductance_value = 0.1

icm = db_client.get_entity(ion_channel_model_ids[0], entity_type=IonChannelModel)
if icm.conductance_name is not None:
    ion_channel_model = IonChannelModelWithConductance(
        ion_channel_model=IonChannelModelFromID(id_str=ion_channel_model_ids[0]),
        conductance=conductance_value,
    )
elif icm.max_permeability_name is not None:
    ion_channel_model = IonChannelModelWithMaxPermeability(
        ion_channel_model=IonChannelModelFromID(id_str=ion_channel_model_ids[0]),
        max_permeability=conductance_value,
    )
else:
    ion_channel_model = IonChannelModelWithoutConductance(
        ion_channel_model=IonChannelModelFromID(id_str=ion_channel_model_ids[0]),
    )

Below is a small piece of code that should find the appropriate variable to record. This should work for most models. If this does not work, you will have to manually input the variable to record value.

In [ ]:
if icm.neuron_block.nonspecific:
    variable_to_record = f"{icm.neuron_block.nonspecific[0]}_{icm.nmodl_suffix}"
elif icm.neuron_block.useion and icm.neuron_block.useion[0].write:
    write = icm.neuron_block.useion[0].write[0]
    variable_to_record = f"{write}_{icm.nmodl_suffix}" if write[0] == "i" or write[0] == "l" else write[0]
else:
    raise ValueError("Could not determine variable to record from ion channel model. Please check the ion channel model's neuron block for either a nonspecific or useion write variable.")

print(f"Recording variable: {variable_to_record}")

In [13]:
sim_conf = IonChannelModelSimulationScanConfig(
    initialize=IonChannelModelSimulationScanConfig.Initialize(
            simulation_length=200,
            temperature=35,
            v_init=-80,
            random_seed=1,
        ),
    info=Info(
        campaign_name="Ion Channel Simulation Campaign Test 001",
        campaign_description="Test",
    ),
    ion_channel_models={
        "icm1": ion_channel_model,
    },
    stimuli={
        "seclamp2": MultiLevelSEClampSomaticStimulus(
            duration_voltage=[
                DurationVoltageCombination(duration=50, voltage=-80),
                DurationVoltageCombination(duration=[100], voltage=[-40, -20, 0, 20, 40, 60]),
                DurationVoltageCombination(duration=30, voltage=-30),
                DurationVoltageCombination(duration=50, voltage=-80),
            ]
        )
    },
    recordings={
        "rec1": IonChannelVariableRecording(
            variable=IonChannelVariableForRecording(
                ion_channel_id=ion_channel_model_ids[0],
                variable_name=variable_to_record,
            ),
            neuron_set=None,
            dt=0.1,
        ),
    },
    timestamps={}
)

# Validated Config
validated_sim_conf = sim_conf.validated_config()

In [ ]:
grid_scan = obi.GridScanGenerationTask(form=validated_sim_conf, coordinate_directory_option="ZERO_INDEX", output_root='../../../obi-output/icm_simulations/grid_scan')
grid_scan.multiple_value_parameters(display=True)
grid_scan.coordinate_parameters(display=True)
grid_scan.execute(db_client=db_client)
obi.run_tasks_for_generated_scan(grid_scan, db_client=db_client)